# scalper-hft: Дослідницький інтерактивний хаб (Quantitative Research Hub)

Інтерактивне середовище для повного циклу кількісних досліджень (R&D) за методологією **Rishi Narang** (*Inside the Black Box*) та **Marcos López de Prado** (*Advances in Financial Machine Learning*).

### Основні розділи ноутбука:
1. **Налаштування оточення**: імпорти, конфігурація та параметри комісій.
2. **Робота з даними**: OHLCV, мікроструктура (aggTrades / CVD) та ставки фінансування.
3. **Бектести та візуалізація**: одиночні стратегії, крива капіталу, підводні просадки, генерація інтерактивних HTML-графіків.
4. **Статистичний парний арбітраж**: спреди, коінтеграція, Maker-виконання та фільтр Калмана.
5. **Аудит на перенавчання (Anti-Overfitting)**: Walk-Forward, Deflated Sharpe Ratio (DSR), CSCV / PBO.
6. **Оптимізація гіперпараметрів**: Optuna TPE з Purged Cross-Validation.
7. **Машинне навчання та важливість фіч**: Triple-Barrier labeling, LightGBM, MDI/MDA/CFI та децильний Lift-аналіз.
8. **Стрес-тести та ємність**: сценарії шоків ринку, масштабування капіталу та time-decay.
9. **Черга задач (Jobs Queue) та фонові воркери**: асинхронний бекенд на SQLite (`results/jobs.sqlite`).
10. **Матричний скан (Matrix Sweep)**: багатовимірний скан сітки (Стратегії × Символи × Таймфрейми).

---
> [!IMPORTANT]
> Усі бектести в цій системі суворо враховують реальні комісії Binance (Maker 0.02% / Taker 0.05%), прослизання (`slippage_frac`), а також виконуються **без lookahead-bias** (сигнал на барі $t$, виконання на барі $t+1$).

In [ ]:
import logging
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Додавання кореня проекту до sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

logging.basicConfig(level=logging.WARNING)

from scalper_hft.config import get_settings
settings = get_settings()

print("=== Конфігурація scalper-hft ===")
print(f"Біржа:         {settings.exchange}")
print(f"Директорія:    {settings.data_dir_abs}")
print(f"Maker комісія: {settings.maker_fee:.4%} (Binance USDT-M)")
print(f"Taker комісія: {settings.taker_fee:.4%}")
print(f"Slippage:      {settings.slippage_frac:.4%}")
print(f"Position %:    {settings.position_pct:.1%}")

## 1. Робота з ринковими даними (Data Access & Exploration)

Завантаження історичних свічок (Klines) з автоматичним ресемплінгом з 1m бази, мікроструктури (aggTrades) та ставок фінансування (Funding Rates).

In [ ]:
from scalper_hft.data.access import ensure_klines

symbol = "BTCUSDT"
interval = "1h"
days = 90

# Завантаження / читання з Parquet-кешу
df = ensure_klines(symbol, interval, days=days, base_interval="1m", derive=True)
print(f"Завантажено свічок: {len(df):,} барів | Діапазон: {df.index[0]} → {df.index[-1]}")
display(df.tail(3))

In [ ]:
from scalper_hft.data.downloader import download_agg_trades, download_funding
from scalper_hft.features.indicators import cvd_from_trades

# 1. Ставки фінансування (Funding Rates)
funding_df = download_funding(symbol, days=30)
if funding_df is not None and not funding_df.empty:
    print(f"Funding Rates ({symbol}): {len(funding_df)} записів, середня ставка: {funding_df['funding_rate'].mean():.6f}")

# 2. Мікроструктура: Cumulative Volume Delta (CVD) з aggTrades
try:
    trades = download_agg_trades(symbol, days=1)
    if trades is not None and not trades.empty:
        cvd = cvd_from_trades(trades, "1min")
        fig, ax = plt.subplots(figsize=(12, 3))
        cvd["cvd"].plot(ax=ax, title=f"Cumulative Volume Delta (CVD) — {symbol}", color="teal", lw=1.5)
        ax.set_ylabel("Контракти")
        ax.grid(True, alpha=0.3)
        plt.show()
except Exception as e:
    print(f"Примітка щодо aggTrades: {e} (завантажте через CLI --trades)")

In [ ]:
# Корисні CLI команди для роботи з даними (за потреби зніміть коментар #):
# !uv run python -m scalper_hft.cli download --symbol BTCUSDT --interval 1h --days 90
# !uv run python -m scalper_hft.cli download --symbol BTCUSDT --interval 1m --days 2 --trades --trades-days 2
# !uv run python -m scalper_hft.cli download --symbol BTCUSDT --interval 1h --days 90 --funding
# !uv run python -m scalper_hft.cli record-bookticker --symbol BTCUSDT --minutes 60
# !uv run python -m scalper_hft.cli migrate-to-parquet --symbol BTCUSDT,ETHUSDT

## 2. Бектести та візуалізація стратегій (Backtesting & Plotting)

Моделювання торгівлі з урахуванням моделі комісій (`CostModel`), розміру позиції та перевіркою метрик (Sharpe, Max Drawdown, Win Rate, Profit Factor).

In [ ]:
from scalper_hft.backtest.engine import run_backtest
from scalper_hft.backtest.execution import CostModel
from scalper_hft.strategies import get_strategy

# Ініціалізація стратегії та моделі витрат
strat = get_strategy("mean_reversion", rsi_period=7, oversold=35, overbought=65, bb_period=20)
cost = CostModel(maker_fee=settings.maker_fee, taker_fee=settings.taker_fee, slippage_frac=settings.slippage_frac)

# Запуск бектесту
res = run_backtest(df, strat, cost=cost, position_pct=settings.position_pct)
print(res.summary())

# Побудова графіків: крива капіталу та підводна просадка (Drawdown)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
res.equity.plot(ax=ax1, title=f"Крива капіталу: {strat.name} ({symbol}, {interval})", color="#1f77b4", lw=1.5)
ax1.set_ylabel("Капітал ($)")
ax1.grid(True, alpha=0.3)

peak = res.equity.cummax()
dd = (res.equity - peak) / peak
dd.plot(ax=ax2, color="#d62728", lw=1.2, title="Просадка (Drawdown)")
ax2.fill_between(dd.index, dd, 0, color="#d62728", alpha=0.25)
ax2.set_ylabel("Drawdown %")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# CLI команди бектесту та побудови інтерактивного HTML-графіка:
# !uv run python -m scalper_hft.cli backtest --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90
# !uv run python -m scalper_hft.cli backtest --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --breakeven-gate
# !uv run python -m scalper_hft.cli plot --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --out docs/plots/backtest.html

## 3. Статистичний парний арбітраж (Statistical Pairs Arbitrage)

Торгівля дельта-нейтральним спредом між двома активами (наприклад XRP/BTC або ETH/BTC) із Maker-виконанням (post-only) та можливістю динамічного розрахунку коефіцієнта хеджування через фільтр Калмана.

In [ ]:
from scalper_hft.backtest.pairs import run_pairs_backtest

leg1 = "XRPUSDT"
leg2 = "BTCUSDT"

d1 = ensure_klines(leg1, "1h", days=90)
d2 = ensure_klines(leg2, "1h", days=90)
f1 = download_funding(leg1, days=90)
f2 = download_funding(leg2, days=90)

pairs_strat = get_strategy("pairs_arb", z_entry=2.0, z_exit=0.5, lookback=120)
pairs_res = run_pairs_backtest(
    d1, d2, pairs_strat, f1, f2,
    position_pct=0.1,
    cost=cost,
    maker_execution=True
)

print("=== Результати бектесту парного арбітражу ===")
print(pairs_res.summary())

fig, ax = plt.subplots(figsize=(12, 4))
pairs_res.equity.plot(ax=ax, title=f"Pairs Arbitrage Equity: {leg1} / {leg2} (Maker execution)", color="darkgreen", lw=1.5)
ax.set_ylabel("Капітал ($)")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# CLI команди для парного арбітражу:
# !uv run python -m scalper_hft.cli coint-scan --symbols BTCUSDT,ETHUSDT,SOLUSDT,XRPUSDT --interval 1h --days 90
# !uv run python -m scalper_hft.cli pairs --strategy pairs_arb --leg1 XRPUSDT --leg2 BTCUSDT --interval 1h --days 90 --maker
# !uv run python -m scalper_hft.cli pairs --strategy pairs_arb --leg1 ETHUSDT --leg2 BTCUSDT --interval 1h --days 90 --maker --use-kalman
# !uv run python -m scalper_hft.cli pairs-portfolio --interval 1h --days 90 --maker --method erc

## 4. Комплексний аудит на перенавчання (Anti-Overfitting Audit)

Захист від data-snooping bias та p-hacking за стандартами **Marcos López de Prado**:
- **Walk-Forward аналіз**: середній OOS Sharpe > 0.3, частка прибуткових вікон $\ge$ 50%.
- **Deflated Sharpe Ratio (DSR)**: перевірка гіпотези з поправкою на кількість перевірених конфігурацій (поріг > 0.95).
- **CSCV / PBO**: ймовірність перенавчання бектесту (PBO < 0.50).

In [ ]:
from scalper_hft.validation.deflated_sharpe import deflated_sharpe_ratio, estimate_n_trials
from scalper_hft.validation.walk_forward import run_walk_forward

# 1. Walk-Forward валідація (In-Sample vs Out-of-Sample вікна)
wf_res = run_walk_forward(df, strat, train_bars=1500, test_bars=500, cost=cost)
print("=== Walk-Forward Аналіз ===")
print(wf_res.summary())

# 2. Deflated Sharpe Ratio (DSR)
rets = res.equity.pct_change().dropna().values
n_trials = estimate_n_trials(len(strat.param_space), 50)
dsr = deflated_sharpe_ratio(rets, n_trials=n_trials)
print(f"\nDeflated Sharpe Ratio: {dsr:.4f} (при оцінці n_trials={n_trials})")
if dsr > 0.95:
    print("✅ Результат: СТАТИСТИЧНО ЗНАЧУЩИЙ EDGE (DSR > 0.95)")
else:
    print("❌ Результат: ПЕРЕНАВЧАННЯ / ШУМ (DSR <= 0.95)")

In [ ]:
# CLI команди повного аудиту на перенавчання:
# !uv run python -m scalper_hft.cli overfit --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --trials 50
# !uv run python -m scalper_hft.cli cscv --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --variants 30 --blocks 8
# !uv run python -m scalper_hft.cli report --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90

## 5. Оптимізація гіперпараметрів (Optuna & Purged K-Fold)

Пошук параметрів за допомогою байєсівської оптимізації (TPE) із Purged/Embargo Cross-Validation.

In [ ]:
# CLI команди оптимізації параметрів:
# !uv run python -m scalper_hft.cli optimize --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --trials 60 --splits 4
# !uv run python -m scalper_hft.cli ml-opt --symbol BTCUSDT --interval 5m --days 30 --trials 40 --splits 5 --embargo 0.01

## 6. Машинне навчання та важливість фіч (AFML)

Розбиття за методом Triple-Barrier, навчання градієнтного бустингу (LightGBM) та аналіз важливості фіч (MDI, MDA, Clustered Feature Importance - CFI).

In [ ]:
# CLI команди ML та аналізу фіч:
# !uv run python -m scalper_hft.cli ml --symbol BTCUSDT --interval 5m --days 30 --pt 1.0 --sl 1.0 --holding 10
# !uv run python -m scalper_hft.cli featimp --symbol BTCUSDT --interval 5m --days 30 --splits 4
# !uv run python -m scalper_hft.cli cfi --symbol BTCUSDT --interval 5m --days 30 --max-clusters 6
# !uv run python -m scalper_hft.cli lift --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --bins 10 --top 10

## 7. Стрес-тести, аналіз ємності та Time-Decay

Перевірка стійкості альфи до ринкових шоків (крах ліквідності, спайки волатильності), оцінка масштабування капіталу (Capacity) та чутливості до затримки входу.

In [ ]:
# CLI команди стрес-тестів та ємності:
# !uv run python -m scalper_hft.cli stress --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 180 --scenarios crash,liquidity,vol_spike,funding_shock
# !uv run python -m scalper_hft.cli capacity --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --scales 1,2,5,10,20 --maker
# !uv run python -m scalper_hft.cli time-decay --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --max-lag 3
# !uv run python -m scalper_hft.cli survival --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90

## 8. Черга задач (Jobs Queue) та фонові воркери

Система scalper-hft підтримує асинхронну чергу задач на **SQLite** (`results/jobs.sqlite`). Задачі ставляться миттєво з дедуплікацією за хешем, а фонові воркери (`job worker`) виконують їх у незалежних процесах.

In [ ]:
from scalper_hft.research.jobs import JobStore

with JobStore() as store:
    jobs = store.list_jobs(limit=10)
    worker_alive = store.worker_is_alive()
    print(f"Статус фонового воркера: {'🟢 ЖИВИЙ' if worker_alive else '🔴 НЕ ЗАПУЩЕНИЙ'}")
    if jobs:
        data = [
            {
                "ID": j.id,
                "Тип": j.kind,
                "Статус": j.status,
                "Хеш": j.short_fp,
                "Прогрес": f"{j.progress_done}/{j.progress_total}" if j.progress_total else "-",
                "Помилка": (j.error or "")[:40],
            }
            for j in jobs
        ]
        display(pd.DataFrame(data))
    else:
        print("Черга задач наразі порожня.")

In [ ]:
# Управління чергою задач та воркерами:
# 1. Поставити бектест у чергу (миттєве повернення):
# !uv run python -m scalper_hft.cli backtest --strategy mean_reversion --symbol BTCUSDT --interval 1h --days 90 --enqueue

# 2. Поставити парний арбітраж у чергу:
# !uv run python -m scalper_hft.cli pairs --strategy pairs_arb --leg1 XRPUSDT --leg2 BTCUSDT --interval 1h --days 90 --maker --enqueue

# 3. Список задач у черзі:
# !uv run python -m scalper_hft.cli job list

# 4. Статус та лог задачі за її ID:
# !uv run python -m scalper_hft.cli job status 1

# 5. Скасувати / перезапустити задачу:
# !uv run python -m scalper_hft.cli job cancel 1
# !uv run python -m scalper_hft.cli job rerun 1

# 6. Очистити завершені задачі та звільнити диск (вік > 14 днів):
# !uv run python -m scalper_hft.cli job prune --days 14

## 9. Матричний скан (Matrix Sweep)

Матричний прогін по сітці (Стратегії × Символи × Таймфрейми). 1-хвилинні базові дані завантажуються один раз на символ, після чого розпаралелюються та ресемпляться на цільові таймфрейми.

In [ ]:
# Запуск матричного скану:
# !uv run python -m scalper_hft.cli sweep --symbols BTCUSDT,ETHUSDT,SOLUSDT --intervals 15m,1h,4h --days 60 --workers 4 --out results/sweep.csv

# Відправити матричний скан у фонову чергу:
# !uv run python -m scalper_hft.cli sweep --symbols BTCUSDT,ETHUSDT --intervals 1h --days 90 --enqueue